In [ ]:

#====================================================================================
# Multi Team/Season Pull
#====================================================================================

#what team(s) and season(s) do you want to pull
season_table = pl.DataFrame(
    {
        "triCode": ["MIN"],
        "seasonId": [20252026], 
    }
)

#======== call APIs ========

#pull games for the teams and seasons
game_list = pull_games(season_table, 2) # 1 pre season // 2 regular season // 3 playoffs // 4 = all stars

# filter to unique games and games that are finished
game_list = (
    game_list
    .unique(subset=["gameId"])
    .filter(pl.col("gameOutcome_lastPeriodType") != "")
)


gameIds = game_list.get_column("gameId").unique().to_list()



#pull play by plays for all games
pbp_all = pl.concat(
    [
        call_play_by_play_api(gid).with_columns(pl.lit(gid).alias("gameId"))
        for gid in gameIds
    ],
    how="diagonal",
)

#pull shifts for all games
shifts_all = pl.concat(
    [
        call_shift_api(str(gid)).with_columns(pl.lit(str(gid)).alias("gameId"))
        for gid in gameIds
    ],
    how="diagonal",
)


#pull players from all seasons/teams selected
players_all = pl.concat(
    [
        call_players_api(str(season_id)).with_columns(
            pl.lit(str(season_id)).alias("season_id")
        )
        for season_id in season_table.get_column("seasonId").unique().to_list()
    ],
    how="diagonal",
)




#======== Summary on data pulled ========

print("Number of games pulled: " + str(len(gameIds)))
print("Number of games with play by play data: " + str(len(pbp_all.get_column("gameId").unique().to_list())))
print("Number of games with shift data: " + str(len(shifts_all.get_column("gameId").unique().to_list())))








In [ ]:

# data tables by season Id - will be easy to pull and can presuably just pull one time for historical seasons. 
team_codes_csv = "https://api.nhle.com/stats/rest/en/team" #static saved as csv
games = "https://api.nhle.com/stats/rest/en/game/summary?&cayenneExp=season=20232024" #teams, dates, scores
skater_summary = "https://api.nhle.com/stats/rest/en/skater/summary?limit=-1&sort=points&cayenneExp=seasonId=20252026"
team_stats = "https://api.nhle.com/stats/rest/en/team/summary?sort=shotsForPerGame&cayenneExp=seasonId=20232024%20and%20gameTypeId=2" #think gametypeid=2 is for reg season and 3 is for playoffs?

play_by_play = "https://api-web.nhle.com/v1/gamecenter/2023020204/play-by-play" #not in a table, but plays are so need code to extract that part of it
shift_chart = "https://api.nhle.com/stats/rest/en/shiftcharts?cayenneExp=gameId=2023020204"
# probs steal the youtubes guys process for putting this together

#2025020693

#reg_season_games.write_excel("_peek.xlsx", autofit=True); __import__("os").startfile("_peek.xlsx")

In [ ]:


# Shifts
SHIFT_SCHEMA = {
    "id": pl.Int64,
    "gameId": pl.Int64,
    "playerId": pl.Int64,
    "period": pl.Int64,
    "shiftNumber": pl.Int64,

    "startTime": pl.Utf8,
    "endTime": pl.Utf8,
    "duration": pl.Utf8,

    # troublemakers: keep as text
    "detailCode": pl.Utf8,
    "typeCode": pl.Utf8,
    "teamId": pl.Utf8,
    "teamAbbrev": pl.Utf8,

    "teamName": pl.Utf8,
    "firstName": pl.Utf8,
    "lastName": pl.Utf8,

    "eventNumber": pl.Int64,
    "eventDescription": pl.Utf8,
    "eventDetails": pl.Utf8,
    "hexValue": pl.Utf8,
}

def empty_df(schema: dict[str, pl.DataType]) -> pl.DataFrame:
    return pl.DataFrame({k: pl.Series([], dtype=v) for k, v in schema.items()})

def normalize_schema(df: pl.DataFrame, schema: dict[str, pl.DataType]) -> pl.DataFrame:
    # add missing cols as nulls with correct dtype
    for col, dtype in schema.items():
        if col not in df.columns:
            df = df.with_columns(pl.lit(None).cast(dtype).alias(col))

    # cast existing cols (strict=False prevents hard failures; bad casts -> null)
    df = df.with_columns([
        pl.col(col).cast(dtype, strict=False)
        for col, dtype in schema.items()
        if col in df.columns
    ])

    return df.select(list(schema.keys()))

# returns homeTeam, awayTeam
def get_team_info(gameId) -> List[str]:
    pbp = call_play_by_play_api(gameId)

    teams = pbp.select(
        pl.col("homeTeamId"),
        pl.col("awayTeamId"),
    )

    homeTeam = teams.get_column("homeTeamId").cast(pl.Utf8)[0]
    awayTeam = teams.get_column("awayTeamId").cast(pl.Utf8)[0]

    return homeTeam, awayTeam



def get_lines_detail(shift_chart, role):

    shifts = (
        shift_chart
        .filter(
            pl.col("roleCode") == role,
        )
    )

    sh = (
        shifts
        .with_columns(
            start_sec = mmss_to_sec(pl.col("startTime")),
            end_sec   = mmss_to_sec(pl.col("endTime")),
        )
        .with_columns(
            abs_start = (pl.col("period") - 1) * 20*60 + pl.col("start_sec"),
            abs_end   = (pl.col("period") - 1) * 20*60 + pl.col("end_sec"),
        )
        .filter(
            pl.col("abs_end") > pl.col("abs_start")
        )

    )

    times = (
        pl.concat([
            sh.select(pl.col("abs_start").alias("t")),
            sh.select(pl.col("abs_end").alias("t")),
        ])
        .unique()
        .sort("t")
    )

    segments = (
        times
        .with_columns(t_next = pl.col("t").shift(-1))
        .drop_nulls()
        .with_columns(
            segment_id = pl.int_range(0, pl.len(), eager=False),
            seg_start  = pl.col("t"),
            seg_end    = pl.col("t_next"),
            seg_dur    = (pl.col("t_next") - pl.col("t")).cast(pl.Int64),
        )
        .filter(pl.col("seg_dur") > 0)
    )

    active = (
        segments
        .join(sh, how="cross")
        .filter(
            (pl.col("abs_start") <= pl.col("seg_start")) &
            (pl.col("seg_start") < pl.col("abs_end"))
        )
        .select([
            "segment_id",
            "seg_start",
            "seg_end",
            "seg_dur",
            "teamId",
            "playerId",
            "skaterFullName",
            "roleCode",
            "positionCode"
        ])
    )

    strength = (
        active
        .group_by(["segment_id", "seg_start", "seg_end"])
        .agg(
            seg_dur = pl.first("seg_dur"),
            teams = pl.col("teamId").unique(),
            counts = pl.col("teamId").value_counts(sort=False),
        )
    )

    full_lines = (
        active
        .join(strength.select("segment_id"), on="segment_id", how="inner")
    )

    return full_lines

def get_line_stints(shift_chart: pl.DataFrame, role: str) -> pl.DataFrame:
    # role: "F" (3 skaters) or "D" (2 skaters)
    if role not in {"F", "D"}:
        raise ValueError("role must be 'F' or 'D'")

    expected = 3 if role == "F" else 2

    full_role_lines = get_lines_detail(shift_chart, role)

    # 1) one row per segment with the combo string
    per_segment = (
        full_role_lines
        .filter(pl.col("roleCode") == role)
        .group_by(["teamId", "segment_id"])
        .agg(
            seg_start = pl.first("seg_start"),
            seg_end = pl.first("seg_end"),
            seg_dur = pl.first("seg_dur"),
            skaters = pl.col("skaterFullName").unique().sort(),
        )
        .filter(pl.col("skaters").list.len() == expected)
        .with_columns(
            line = pl.col("skaters").list.join(" - ")
        )
        .select(["teamId", "segment_id", "seg_start", "seg_end", "seg_dur", "line"])
        .sort(["teamId", "seg_start"])
    )

    # 2) collapse consecutive segments with same line into stints (per team)
    stints = (
        per_segment
        .with_columns(
            is_new_stint = (pl.col("line") != pl.col("line").shift(1))
                .over("teamId")
                .fill_null(True)
        )
        .with_columns(
            stint_num = pl.col("is_new_stint")
                .cast(pl.Int64)
                .over("teamId")
                .cum_sum()
        )
        .group_by(["teamId", "stint_num", "line"])
        .agg(
            stint_start = pl.min("seg_start"),
            stint_end = pl.max("seg_end"),
            duration_sec = pl.sum("seg_dur"),
            segment_count = pl.len(),
        )
        .with_columns(roleCode = pl.lit(role))
        .sort(["teamId", "stint_start"])
    )

    # 3) add readable period/MM:SS start/end
    start_period, start_mmss = sec_to_period_mmss(pl.col("stint_start"))
    end_period, end_mmss = sec_to_period_mmss(pl.col("stint_end"))

    stints = stints.with_columns(
        startPeriod = start_period,
        startTime = start_mmss,
        endPeriod = end_period,
        endTime = end_mmss,
    )

    return stints

def get_lines_combos(shift_chart: pl.DataFrame, role: str) -> pl.DataFrame:
    # role: "F" (3 skaters) or "D" (2 skaters)
    if role not in {"F", "D"}:
        raise ValueError("role must be 'F' or 'D'")

    expected = 3 if role == "F" else 2

    full_role_lines = get_lines_detail(shift_chart, role)

    per_segment = (
        full_role_lines
        .filter(pl.col("roleCode") == role)
        .group_by(["teamId", "segment_id"])
        .agg(
            seg_start = pl.first("seg_start"),
            seg_end   = pl.first("seg_end"),
            seg_dur   = pl.first("seg_dur"),
            skaters   = pl.col("skaterFullName").unique().sort(),
        )
        .filter(pl.col("skaters").list.len() == expected)
        .with_columns(
            line = pl.col("skaters").list.join(" - ")
        )
        .select(["teamId", "segment_id", "seg_start", "seg_end", "seg_dur", "line"])
        .sort(["teamId", "seg_start"])
    )

    # mark new stint when line changes (within team)
    per_segment = (
        per_segment
        .with_columns(
            is_new_stint = (pl.col("line") != pl.col("line").shift(1))
                .over("teamId")
                .fill_null(True)
        )
        .with_columns(
            stint_id = pl.col("is_new_stint")
                .cast(pl.Int64)
                .over("teamId")
                .cum_sum()
        )
    )

    unique_lines = (
        per_segment
        .group_by(["teamId", "line"])
        .agg(
            total_sec     = pl.sum("seg_dur"),
            total_min     = (pl.sum("seg_dur") / 60).round(2),
            segment_count = pl.len(),
            stint_count   = pl.col("stint_id").n_unique(),
        )
        .sort(["teamId", "total_sec"], descending=[False, True])
    )

    return unique_lines


# returns list [home_fwd_stints, home_def_stints, away_fwd_stints, away_def_stints]
def get_game_shifts(gameId) -> List[pl.DataFrame]:

    homeTeam = get_team_info(gameId)[0]
    awayTeam = get_team_info(gameId)[1]

     
    seasonId = "20252026" # code this out to pull the first four digits of game id and calc seasonId

    shifts = get_shift_detail(gameId, seasonId)

    home_shifts = shifts.filter(pl.col("teamId") == homeTeam)
    home_fwd_stints = get_line_stints(home_shifts,"F")
    home_def_stints = get_line_stints(home_shifts,"D")

    away_shifts = shifts.filter(pl.col("teamId") == awayTeam)   
    away_fwd_stints = get_line_stints(away_shifts,"F")
    away_def_stints = get_line_stints(away_shifts,"D")

    return [home_fwd_stints, home_def_stints, away_fwd_stints, away_def_stints]



def get_matchups(home_stints: pl.DataFrame, away_stints: pl.DataFrame) -> pl.DataFrame: # filter to minimum overlap > 35 seconds
    h = (
        home_stints
        .select([
            pl.col("teamId").alias("homeTeamId"),
            pl.col("stint_num").alias("home_stint_num"),
            pl.col("line").alias("home_line"),
            pl.col("stint_start").alias("home_start"),
            pl.col("stint_end").alias("home_end"),
        ])
    )

    a = (
        away_stints
        .select([
            pl.col("teamId").alias("awayTeamId"),
            pl.col("stint_num").alias("away_stint_num"),
            pl.col("line").alias("away_line"),
            pl.col("stint_start").alias("away_start"),
            pl.col("stint_end").alias("away_end"),
        ])
    )

    # Pair up stints and keep only those that overlap in time
    overlaps = (
        h.join(a, how="cross")
        .filter(
            (pl.col("home_start") < pl.col("away_end")) &
            (pl.col("home_end") > pl.col("away_start"))
        )
        .with_columns(
            overlap_start = pl.max_horizontal("home_start", "away_start"),
            overlap_end   = pl.min_horizontal("home_end", "away_end"),
        )
        .with_columns(
            overlap_sec = (pl.col("overlap_end") - pl.col("overlap_start")).cast(pl.Int64)
        )
        .filter(pl.col("overlap_sec") > overlap_duration)
    )

    # Aggregate to line-vs-line totals
    matchup_totals = (
        overlaps
        .group_by(["homeTeamId", "awayTeamId", "home_line", "away_line"])
        .agg(
            overlap_sec = pl.sum("overlap_sec"),
            overlap_min = (pl.sum("overlap_sec") / 60).round(2),
            overlap_count = pl.len(),  # number of overlapping stint-pairs
            first_overlap_start = pl.min("overlap_start"),
            last_overlap_end    = pl.max("overlap_end"),
        )
        .sort("overlap_sec", descending=True)
    )

    return matchup_totals

def get_matchup_instances(home_stints: pl.DataFrame, away_stints: pl.DataFrame) -> pl.DataFrame:
    h = home_stints.select([
        pl.col("teamId").alias("homeTeamId"),
        pl.col("stint_num").alias("home_stint_num"),
        pl.col("line").alias("home_line"),
        pl.col("stint_start").alias("home_start"),
        pl.col("stint_end").alias("home_end"),
        pl.col("startPeriod").alias("home_startPeriod"),
        pl.col("startTime").alias("home_startTime"),
        pl.col("endPeriod").alias("home_endPeriod"),
        pl.col("endTime").alias("home_endTime"),
    ])

    a = away_stints.select([
        pl.col("teamId").alias("awayTeamId"),
        pl.col("stint_num").alias("away_stint_num"),
        pl.col("line").alias("away_line"),
        pl.col("stint_start").alias("away_start"),
        pl.col("stint_end").alias("away_end"),
        pl.col("startPeriod").alias("away_startPeriod"),
        pl.col("startTime").alias("away_startTime"),
        pl.col("endPeriod").alias("away_endPeriod"),
        pl.col("endTime").alias("away_endTime"),
    ])

    return (
        h.join(a, how="cross")
        .filter((pl.col("home_start") < pl.col("away_end")) & (pl.col("home_end") > pl.col("away_start")))
        .with_columns(
            overlap_start = pl.max_horizontal("home_start", "away_start"),
            overlap_end = pl.min_horizontal("home_end", "away_end"),
        )
        .with_columns(
            overlap_sec = (pl.col("overlap_end") - pl.col("overlap_start")).cast(pl.Int64),
            overlap_min = ((pl.col("overlap_end") - pl.col("overlap_start")) / 60).round(2),
        )
        .filter(pl.col("overlap_sec") > 0)
        .sort("overlap_sec", descending=True)
    )


def build_pbp_with_on_ice(pbp: pl.DataFrame, shifts: pl.DataFrame) -> pl.DataFrame:


    # --- PBP absolute event second ---
    pbp = pbp.with_columns(
        event_sec=(pl.col("periodNumber") - 1) * 20 * 60 + mmss_to_sec(pl.col("timeInPeriod"))
    )

    # --- Clean shift intervals in absolute seconds ---
    shifts_clean = (
        shifts
        .with_columns(
            start_sec=mmss_to_sec(pl.col("startTime")),
            end_sec=mmss_to_sec(pl.col("endTime")),
        )
        .with_columns(
            abs_start=(pl.col("period") - 1) * 20 * 60 + pl.col("start_sec"),
            abs_end=(pl.col("period") - 1) * 20 * 60 + pl.col("end_sec"),
        )
        .filter(pl.col("abs_end") > pl.col("abs_start"))
    )

    # Prefer dropping goalies by positionCode if present
    if "positionCode" in shifts_clean.columns:
        shifts_clean = shifts_clean.filter(pl.col("positionCode") != "G")

    shifts_clean = shifts_clean.select(["teamId", "playerId", "abs_start", "abs_end"])

    # --- Build on-ice by second (skaters only) ---
    on_ice_by_sec = (
        shifts_clean
        .with_columns(sec=pl.int_ranges("abs_start", "abs_end", step=1))
        .explode("sec")
        .group_by(["teamId", "sec"])
        .agg(on_ice=pl.col("playerId").unique())
        .sort(["teamId", "sec"])
    )

    # --- Asof join (backward) to handle boundary seconds (e.g., OT goal at end) ---
    events = pbp.select(["eventId", "event_sec", "homeTeamId", "awayTeamId"]).unique()

    home_events = (
        events
        .with_columns(teamId=pl.col("homeTeamId"))
        .sort(["teamId", "event_sec"])  # REQUIRED for join_asof correctness
    )

    away_events = (
        events
        .with_columns(teamId=pl.col("awayTeamId"))
        .sort(["teamId", "event_sec"])  # REQUIRED for join_asof correctness
    )

    home_on = (
        home_events
        .join_asof(
            on_ice_by_sec,
            left_on="event_sec",
            right_on="sec",
            by="teamId",
            strategy="backward",
        )
        .rename({"on_ice": "home_on_ice", "sec": "home_match_sec"})
        .select(["eventId", "home_on_ice", "home_match_sec"])
    )

    away_on = (
        away_events
        .join_asof(
            on_ice_by_sec,
            left_on="event_sec",
            right_on="sec",
            by="teamId",
            strategy="backward",
        )
        .rename({"on_ice": "away_on_ice", "sec": "away_match_sec"})
        .select(["eventId", "away_on_ice", "away_match_sec"])
    )

    out = (
        pbp
        .join(home_on, on="eventId", how="left")
        .join(away_on, on="eventId", how="left")
        .with_columns(
            home_lag=pl.col("event_sec") - pl.col("home_match_sec"),
            away_lag=pl.col("event_sec") - pl.col("away_match_sec"),
        )
    )

    # Safety: if it snaps back too far, null it out (normally 0–1; 2 is ok)
    out = out.with_columns(
        home_on_ice=pl.when(pl.col("home_lag") <= 2).then(pl.col("home_on_ice")).otherwise(None),
        away_on_ice=pl.when(pl.col("away_lag") <= 2).then(pl.col("away_on_ice")).otherwise(None),
    )

    return out


# adding GF/GA CF/CA FF/FA

def build_event_player_long(pbp_with_on_ice: pl.DataFrame) -> pl.DataFrame:
    """
    Explodes pbp_with_on_ice into event-player long form.

    - Renames PBP's 'playerId' (event actor) -> 'eventPlayerId' if present
    - Creates 'onIcePlayerId' for the exploded on-ice skater
    - Adds: side ('home'/'away'), teamId (team for that side)
    """

    exclude = {
        "home_on_ice", "away_on_ice",
        "home_match_sec", "away_match_sec",
        "home_lag", "away_lag",
        "homeTeamId", "awayTeamId",  # we add teamId explicitly per side
    }

    base_cols = [c for c in pbp_with_on_ice.columns if c not in exclude]

    # Build select expressions so we can alias PBP 'playerId' if it exists
    base_exprs = []
    for c in base_cols:
        if c == "playerId":
            base_exprs.append(pl.col("playerId").alias("eventPlayerId"))
        else:
            base_exprs.append(pl.col(c))

    home_long = (
        pbp_with_on_ice
        .select(base_exprs + [pl.col("homeTeamId"), pl.col("home_on_ice")])
        .with_columns(
            side=pl.lit("home"),
            teamId=pl.col("homeTeamId"),
        )
        .drop("homeTeamId")
        .explode("home_on_ice")
        .rename({"home_on_ice": "onIcePlayerId"})
    )

    away_long = (
        pbp_with_on_ice
        .select(base_exprs + [pl.col("awayTeamId"), pl.col("away_on_ice")])
        .with_columns(
            side=pl.lit("away"),
            teamId=pl.col("awayTeamId"),
        )
        .drop("awayTeamId")
        .explode("away_on_ice")
        .rename({"away_on_ice": "onIcePlayerId"})
    )

    out = pl.concat([home_long, away_long], how="diagonal")

    # keep ids consistent
    if out.schema.get("onIcePlayerId") != pl.Utf8:
        out = out.with_columns(pl.col("onIcePlayerId").cast(pl.Utf8))
    if "eventPlayerId" in out.columns and out.schema.get("eventPlayerId") != pl.Utf8:
        out = out.with_columns(pl.col("eventPlayerId").cast(pl.Utf8))

    return out

def add_strength_counts(event_player_long: pl.DataFrame) -> pl.DataFrame:
    """
    Adds home_skaters / away_skaters per event based on on-ice lists.
    """

    counts = (
        event_player_long
        .group_by(["eventId", "side"])
        .agg(skaters=pl.len())
        .pivot(
            values="skaters",
            index="eventId",
            columns="side",
        )
        .rename({
            "home": "home_skaters",
            "away": "away_skaters",
        })
    )

    return event_player_long.join(counts, on="eventId", how="left")

def add_strength_flags(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        is_5v5 = ((pl.col("home_skaters") == 5) & (pl.col("away_skaters") == 5)).cast(pl.Int8),
        is_5v4 = (
            ((pl.col("home_skaters") == 5) & (pl.col("away_skaters") == 4)) |
            ((pl.col("home_skaters") == 4) & (pl.col("away_skaters") == 5))
        ).cast(pl.Int8),
    )

def add_for_against_flags(df: pl.DataFrame) -> pl.DataFrame:
    shot_events = ["shot-on-goal", "goal", "missed-shot", "blocked-shot"]
    fenwick_events = ["shot-on-goal", "goal", "missed-shot"]

    return (
        df
        .with_columns(
            is_goal = (pl.col("typeDescKey") == "goal").cast(pl.Int8),
            is_corsi_event = pl.col("typeDescKey").is_in(shot_events).cast(pl.Int8),
            is_fenwick_event = pl.col("typeDescKey").is_in(fenwick_events).cast(pl.Int8),

            # base FOR flag
            base_is_for = (pl.col("eventOwnerTeamId") == pl.col("teamId")),
        )
        .with_columns(
            # FIX: blocked shots belong to the shooting team, not eventOwnerTeamId
            is_for = pl.when(pl.col("typeDescKey") == "blocked-shot")
                        .then(pl.col("eventOwnerTeamId") != pl.col("teamId"))
                        .otherwise(pl.col("base_is_for"))
                        .cast(pl.Int8)
        )
        .with_columns(
            GF = (pl.col("is_goal") & pl.col("is_for")).cast(pl.Int8),
            GA = (pl.col("is_goal") & (~pl.col("is_for").cast(bool))).cast(pl.Int8),

            CF = (pl.col("is_corsi_event") & pl.col("is_for")).cast(pl.Int8),
            CA = (pl.col("is_corsi_event") & (~pl.col("is_for").cast(bool))).cast(pl.Int8),

            FF = (pl.col("is_fenwick_event") & pl.col("is_for")).cast(pl.Int8),
            FA = (pl.col("is_fenwick_event") & (~pl.col("is_for").cast(bool))).cast(pl.Int8),
        )
        .drop("base_is_for")
    )

def build_player_team_map(shifts: pl.DataFrame) -> pl.DataFrame:
    return (
        shifts
        .select(["playerId", "teamId"])
        .unique(subset=["playerId"])
    )

def build_event_team_attribution_from_shooter(
    pbp_with_on_ice: pl.DataFrame,
    shifts: pl.DataFrame,
) -> pl.DataFrame:
    """
    One row per (eventId, teamId) with GF/GA, CF/CA, FF/FA using NST-style attribution:
      - FOR team for shot attempts = shooter's team (via shootingPlayerId -> shifts teamId)
      - fallback = eventOwnerTeamId when shooter team missing
    """

    shot_events = ["shot-on-goal", "goal", "missed-shot", "blocked-shot"]
    fenwick_events = ["shot-on-goal", "goal", "missed-shot"]

    # grab the two teams once (constants)
    home_id = pbp_with_on_ice.get_column("homeTeamId")[0]
    away_id = pbp_with_on_ice.get_column("awayTeamId")[0]

    # map playerId -> teamId from shifts (within this game)
    player_team = shifts.select(["playerId", "teamId"]).unique(subset=["playerId"])

    # minimal pbp view + attach shooter team
    pbp2 = (
        pbp_with_on_ice
        .select(["eventId", "typeDescKey", "eventOwnerTeamId", "shootingPlayerId"])
        .join(
            player_team.rename({"playerId": "shootingPlayerId", "teamId": "shooterTeamId"}),
            on="shootingPlayerId",
            how="left",
        )
        .with_columns(
            shot_for_teamId=pl.coalesce([pl.col("shooterTeamId"), pl.col("eventOwnerTeamId")])
        )
        .select(["eventId", "typeDescKey", "shot_for_teamId"])
    )

    # two rows per event (home+away) using constants
    home_rows = pbp2.with_columns(teamId=pl.lit(home_id))
    away_rows = pbp2.with_columns(teamId=pl.lit(away_id))

    et = pl.concat([home_rows, away_rows], how="vertical")

    et = et.with_columns(
        is_goal=(pl.col("typeDescKey") == "goal").cast(pl.Int8),
        is_corsi_event=pl.col("typeDescKey").is_in(shot_events).cast(pl.Int8),
        is_fenwick_event=pl.col("typeDescKey").is_in(fenwick_events).cast(pl.Int8),
        is_for=(pl.col("teamId") == pl.col("shot_for_teamId")).cast(pl.Int8),
    ).with_columns(
        GF=(pl.col("is_goal") & pl.col("is_for")).cast(pl.Int8),
        GA=(pl.col("is_goal") & (~pl.col("is_for").cast(bool))).cast(pl.Int8),
        CF=(pl.col("is_corsi_event") & pl.col("is_for")).cast(pl.Int8),
        CA=(pl.col("is_corsi_event") & (~pl.col("is_for").cast(bool))).cast(pl.Int8),
        FF=(pl.col("is_fenwick_event") & pl.col("is_for")).cast(pl.Int8),
        FA=(pl.col("is_fenwick_event") & (~pl.col("is_for").cast(bool))).cast(pl.Int8),
    )

    return et.select(["eventId", "teamId", "GF", "GA", "CF", "CA", "FF", "FA"])

def build_player_event_detail(gameId, seasonId) -> pl.DataFrame:
    pbp = call_play_by_play_api(gameId)
    shifts = get_shift_detail(gameId, seasonId)

    pbp_with_on_ice = build_pbp_with_on_ice(pbp, shifts)

    event_player_long = (
        build_event_player_long(pbp_with_on_ice)
        .pipe(add_strength_counts)
        .pipe(add_strength_flags)
    )

    event_team_flags = build_event_team_attribution_from_shooter(pbp_with_on_ice, shifts)

    event_player_long = event_player_long.join(
        event_team_flags,
        on=["eventId", "teamId"],
        how="left",
    )

    return event_player_long



In [ ]:
=SWITCH(O2,0,"","N","N",IF(J2=AQ2,O2,IF(O2="D","O","D")))

In [ ]:

#====================================================================================
# Single Game Pull
#====================================================================================
seasonId = "20252026"

game_list = call_schedule_api(f"https://api-web.nhle.com/v1/club-schedule-season/MIN/{seasonId}") 
game_list.write_excel("_games.xlsx", autofit=True); __import__("os").startfile("_games.xlsx")

players_list = call_players_api(seasonId) 
players_list.write_excel("_players.xlsx", autofit=True); __import__("os").startfile("_players.xlsx")

#vegas "2025020655"
#kraken "2025020693"
#islanders "2025020708"
gameId = "2025020693"

pbp = call_play_by_play_api(gameId)
pbp.write_excel("_pbp.xlsx", autofit=True); __import__("os").startfile("_pbp.xlsx")

shifts = call_shift_api(gameId)

#add shiftNumber for each game/player/period
shifts = (
    shifts
    .with_columns(
        pl.col("startTime")
        .rank("dense")
        .over(["gameId", "playerId", "period"])
        .alias("shiftNumber")
    )
)

shifts.write_excel("_shifts.xlsx", autofit=True); __import__("os").startfile("_shifts.xlsx")




In [ ]:

#======== Data cleaning ========


# only keep columns we need for shifts
shifts = shifts_all.select(
    pl.col("gameId"),
    pl.col("playerId"),
    pl.col("startTime"),
    pl.col("endTime"),
    pl.col("duration"),
    pl.col("period"),
    pl.col("shiftNumber"),
    pl.col("teamId"),
    pl.col("eventDescription"),
)


# add shift number for each game/shift/player
shifts = (
    shifts
    .with_columns(
        pl.col("startTime")
        .rank("dense")
        .over(["gameId", "playerId", "period"])
        .alias("shift"),
        (pl.col("period")+pl.col("shiftNumber")/100).alias("shiftId"),
        pl.col("playerId").cast(pl.Utf8)
    )
)


shifts = (
    shifts
    .join(
        players_all,
        on="playerId",
        how="left",
    )
    
)


In [ ]:
players_list = api.call_players_api(seasonId) 
players_list.write_excel("_players.xlsx", autofit=True); __import__("os").startfile("_players.xlsx")